In [1]:
import pandas as pd
import numpy as np
import re
import networkx as nx
from matplotlib.lines import Line2D
import os
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import matplotlib.pyplot as plt
import pylab as plt
import openpyxl

In [2]:
CD8_SZABO_FILE = "CD8_Szabo_Activation_Markers.csv"
COMPOUND_FILE  = "cd8_limma_merged_filtered_targets_ic50.csv"

cd8_szabo = pd.read_csv(CD8_SZABO_FILE)\
    .rename(columns={"Unnamed: 0": "ensembl", "avg_log2FC": "szabo_lfc"})\
    .dropna(subset=["gene", "szabo_lfc"])\
    .drop_duplicates(subset="gene")[["gene", "szabo_lfc"]]

szabo_lookup = cd8_szabo.set_index("gene")["szabo_lfc"]
print(f"CD8 Szabo genes: {len(cd8_szabo):,}")

df = pd.read_csv(COMPOUND_FILE)\
    .rename(columns={"Unnamed: 0": "sample", "Unnamed: 1": "row_idx"})

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

print(f"Significant genes kept  : {(df_avg['adj_P_Val'] < 0.05).sum():,}")
print(f"Insignificant → zeroed  : {(df_avg['adj_P_Val'] >= 0.05).sum():,}")

pivot = df_avg.pivot_table(
    index   = "compound_name",
    columns = "gene",
    values  = "logFC_thresh",
    aggfunc = "first"
).fillna(0)

CD8 Szabo genes: 10,749
Significant genes kept  : 60,158
Insignificant → zeroed  : 0


In [3]:
# 4. Build STV_activation
#

common_genes  = pivot.columns.intersection(szabo_lookup.index)
pivot_aligned = pivot[common_genes]
szabo_aligned = szabo_lookup[common_genes].values

STV_activation = szabo_aligned / np.linalg.norm(szabo_aligned)

# 5. DPD per compound = dot(compound logFC vector, STV_activation)

dpd_vec = pivot_aligned.values @ STV_activation

dpd_df = pd.DataFrame({
    "compound_name" : pivot_aligned.index,
    "dpd"           : dpd_vec.round(4),
    "direction"     : ["drives_activation" if x > 0 else "drives_resting"
                       for x in dpd_vec]
}).sort_values("dpd", ascending=False).reset_index(drop=True)

# 6. Add targets and mechanism

targets = pd.read_csv(COMPOUND_FILE)[
    ["compound_name", "target_protein", "mechanism"]
].drop_duplicates(subset="compound_name")

dpd_df = dpd_df.merge(targets, on="compound_name", how="left")

# 7. Save
dpd_df.to_csv("dpd_sum_per_compound_raw.csv", index=False)
print("\n✓ Saved dpd_sum_per_compound_raw.csv")



✓ Saved dpd_sum_per_compound_raw.csv


In [4]:
# Compare STV_activation and STV_btla directly, aligned on their own shared genes
# (independent of the compound-pivot alignment used above — kept in separate,
# distinctly-named variables so this cell can't clobber STV_activation/common_genes
# from the DPD cells above, regardless of run order)

szabo_lookup_cmp = pd.read_csv("CD8_Szabo_Activation_Markers.csv")\
    .assign(gene=lambda d: d["gene"].astype(str).str.strip())\
    .dropna(subset=["gene", "avg_log2FC"])\
    .drop_duplicates(subset="gene")\
    .set_index("gene")["avg_log2FC"]

btla_lookup_cmp = pd.read_csv("/CD8_Szabo_Activation_Markers.csv")\
    .assign(gene=lambda d: d["gene"].astype(str).str.strip())\
    .dropna(subset=["gene", "avg_log2FC"])\
    .drop_duplicates(subset="gene")\
    .set_index("gene")["avg_log2FC"]

common_genes_cmp = szabo_lookup_cmp.index.intersection(btla_lookup_cmp.index)

STV_activation_cmp = (szabo_lookup_cmp[common_genes_cmp] / np.linalg.norm(szabo_lookup_cmp[common_genes_cmp])).values
STV_btla_cmp       = (btla_lookup_cmp[common_genes_cmp]  / np.linalg.norm(btla_lookup_cmp[common_genes_cmp])).values

# Dot product / cosine similarity / angle (STVs are already unit length, so dot == cosine similarity)

dot = np.dot(STV_activation_cmp, STV_btla_cmp)
cos_sim = dot
angle_deg = np.degrees(np.arccos(np.clip(cos_sim, -1.0, 1.0)))

print(f"n common genes: {len(common_genes_cmp)}")
print(f"Dot product:        {dot:.6f}")
print(f"Sign:                {'POSITIVE' if dot > 0 else 'NEGATIVE' if dot < 0 else 'ZERO'}")
print(f"Cosine similarity:   {cos_sim:.6f}")
print(f"Angle (degrees):     {angle_deg:.2f}")


n common genes: 10749
Dot product:        1.000000
Sign:                POSITIVE
Cosine similarity:   1.000000
Angle (degrees):     0.00


In [5]:
# Per-gene DPD contribution: logFC_thresh(gene) * STV_activation(gene) [elementwise term
# of the dot product — sums to each compound's total in dpd_sum_per_compound_raw.csv]

gene_dpd = df_avg[df_avg["gene"].isin(common_genes)].copy()
stv_by_gene = pd.Series(STV_activation, index=common_genes)

gene_dpd["dpd"] = gene_dpd["logFC_thresh"] * gene_dpd["gene"].map(stv_by_gene)
gene_dpd["cd8_lfc"] = gene_dpd["gene"].map(szabo_lookup)

# Rank + share of each compound's total DPD
gene_dpd["dpd_rank"] = (
    gene_dpd["dpd"].abs()
    .groupby(gene_dpd["compound_name"])
    .rank(ascending=False, method="first")
)

compound_totals = gene_dpd.groupby("compound_name")["dpd"].transform("sum")
gene_dpd["dpd_score"] = gene_dpd["dpd"] / compound_totals

# Merge in the per-compound metadata (dose/IC50/target/mechanism) from the base file
compound_meta = pd.read_csv("cd8_limma_merged_filtered_targets_ic50.csv")[
    ["compound_name", "gene", "logFC", "adj.P.Val", "dose_uM", "IC50_nM", "target_protein", "mechanism"]
]

out_df = compound_meta.merge(
    gene_dpd[["compound_name", "gene", "dpd", "cd8_lfc", "dpd_rank", "dpd_score"]],
    on=["compound_name", "gene"],
    how="left",
)

out_df.to_csv("cd8_limma_merged_filtered_targets_ic50_dpd.csv", index=False)
print("✓ Saved cd8_limma_merged_filtered_targets_ic50_dpd.csv")


✓ Saved cd8_limma_merged_filtered_targets_ic50_dpd.csv


In [6]:
top4 = dpd_df.nlargest(4, "dpd")
bottom4 = dpd_df.nsmallest(4, "dpd")

top_bottom4 = pd.concat([top4, bottom4]).reset_index(drop=True)
top_bottom4

,compound_name,dpd,direction,target_protein,mechanism
0,Thapsigargin,8.4734,drives_activation,Sarcoplasmic/endoplasmic reticulum calcium ATP...,Inhibitor
1,QS-11,1.3683,drives_activation,ARFGAP1 (ADP-ribosylation factor GTPase-activa...,Inhibitor
2,Pevonedistat,1.1825,drives_activation,NEDD8-activating enzyme (NAE),Inhibitor
3,navitoclax,0.6498,drives_activation,BCL-2; BCL-XL; BCL-W,Inhibitor
4,ruxolitinib,-31.2463,drives_resting,JAK1; JAK2,Inhibitor
5,Sapanisertib,-19.8254,drives_resting,Serine/threonine-protein kinase mTOR,Inhibitor
6,Temsirolimus,-12.3376,drives_resting,Serine/threonine-protein kinase mTOR,Inhibitor
7,BMS-536924,-9.8623,drives_resting,Insulin-like growth factor 1 receptor (IGF-1R),Inhibitor


In [7]:
top_bottom4.to_csv("top4_bottom4_dpd.csv", index=False)
